In [ ]:
import pandas as pd

# CLIENTES (CSV)
clientes = pd.read_csv("../../datasets/segundos_clientes/clientes.csv")
# VENTAS (XLSX)
ventas   = pd.read_excel("../../datasets/segundos_clientes/ventas.xlsx", sheet_name="ventas")
# CONTACTO(JSON)
contacto = pd.read_json("../../datasets/segundos_clientes/contacto.json")

print(clientes)
print(ventas)
print(contacto)

5
4
   cliente_id        producto  cantidad  importe
0        1001  Hotel 3 noches         1    120.5
1        1002           Vuelo         2    180.0
2        1003     Tour guiado         1     45.0
3        1005   Hotel 1 noche         1     80.0
   cliente_id             email
0        1001     ana@email.com
1        1002    luis@email.com
2        1004  carlos@email.com


In [ ]:
registros_antes = len(clientes)
# ELIMINA LOS CLIENTES DUPLICADOS
clientes = clientes.drop_duplicates()
duplicados = registros_antes - len(clientes)

In [8]:
# ELIMINA LOS ESPACIOS DE LOS NOMBRES DE LOS CLIENTES Y PONE LA PRIMERA LETRA EN MAYÚSCULA
clientes["nombre"] = clientes["nombre"].str.strip().str.capitalize()
print(clientes["nombre"])

0       Ana
1      Luis
2     Marta
4    Carlos
Name: nombre, dtype: str


In [9]:
# PONE LA PRIMERA LETRA DE CADA PALABRA EN MAYÚSCULA (POR EJEMPLO: CIUDAD RODRIGO)
clientes["ciudad"] = clientes["ciudad"].str.strip().str.title()
print(clientes["ciudad"])

0        Madrid
1     Salamanca
2    Valladolid
4           NaN
Name: ciudad, dtype: str


In [10]:
# FORMATEO DE LAS FECHAS
clientes["fecha_alta"] = pd.to_datetime(
    clientes["fecha_alta"], format="%d/%m/%Y", errors="coerce"
).dt.strftime("%Y-%m-%d")

print(clientes["fecha_alta"])

0    2024-01-15
1    2024-02-20
2    2024-03-18
4    2024-04-05
Name: fecha_alta, dtype: str


In [ ]:
# RENOMBRAMOS LA CLAVE DE LAS COLUMNAS PARA QUE SE LLAMEN TODAS id_cliente
clientes = clientes.rename(columns={"cliente_id": "id_cliente"})

In [23]:
integrado = clientes.merge(ventas, on="id_cliente", how="left")
integrado = integrado.merge(contacto, on="id_cliente", how="left")

print(integrado)

   id_cliente  nombre      ciudad  fecha_alta        producto  cantidad  \
0        1001     Ana      Madrid  2024-01-15  Hotel 3 noches       1.0   
1        1002    Luis   Salamanca  2024-02-20           Vuelo       2.0   
2        1003   Marta  Valladolid  2024-03-18     Tour guiado       1.0   
3        1004  Carlos         NaN  2024-04-05             NaN       NaN   

   importe             email  
0    120.5     ana@email.com  
1    180.0    luis@email.com  
2     45.0               NaN  
3      NaN  carlos@email.com  


In [ ]:
integrado = integrado.sort_values(["id_cliente", "id_venta"], na_position="last")
for columna in ("id_venta", "cantidad"):
    integrado[columna] = integrado[columna].astype("Int64")

In [25]:
ciudades = clientes["ciudad"].fillna("").str.strip()

incidencias = {
    "duplicados":      clientes.duplicated().sum(),
    "ciudades_vacias": (ciudades == "").sum(),
    "ventas_sin_cliente":      (~ventas["id_cliente"].isin(clientes["id_cliente"])).sum(),
    "clientes_sin_correo":     (~clientes["id_cliente"].isin(contacto["id_cliente"])).sum(),
    "contactos_sin_correspondencia": (~contacto["id_cliente"].isin(clientes["id_cliente"])).sum(),
}

print(incidencias)

{'duplicados': np.int64(0), 'ciudades_vacias': np.int64(1), 'ventas_sin_cliente': np.int64(1), 'clientes_sin_correo': np.int64(1), 'contactos_sin_correspondencia': np.int64(0)}


In [ ]:
texto = f"""====================================
INFORME DE INTEGRACIÓN
====================================

Fecha de ejecución:
{incidencias}
...
Proceso finalizado correctamente.
"""
FICHERO_INFORME.write_text(texto, encoding="utf-8")

In [26]:
#integrado.to_csv(FICHERO_SALIDA, index=False, encoding="utf-8", lineterminator="\n")
integrado.to_csv("../../datasets/segundos_clientes/integrado.csv")
#digest = hashlib.sha256(FICHERO_SALIDA.read_bytes()).hexdigest()

In [ ]:
python etl_integracion.py --check-reproducibilidad